## **Applio**
Простой и высококачественный инструмент для преобразования голоса, ориентированный на простоту использования и производительность.

[Поддержка](https://discord.gg/urxFjYmYYh) — [Discord-бот](https://discord.com/oauth2/authorize?client_id=1144714449563955302&permissions=1376674695271&scope=bot%20applications.commands) — [Найти голоса](https://applio.org/models) — [GitHub](https://github.com/IAHispano/Applio)

<br>

### **Благодарности**
- Метод шифрования: [Hina](https://github.com/hinabl)
- Дополнительный раздел: [Poopmaster](https://github.com/poiqazwsx)
- Основная разработка: [Команда Applio](https://github.com/IAHispano)

In [ ]:
# @title Подключить Google Диск
# @markdown Подключить файлы из Google Диска в Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Запуск вне среды Google Colab. Пропуск подключения Google Диска.")

In [ ]:
# @title **Установка Applio**
import os
import shutil
import tarfile
import subprocess
from pathlib import Path
from IPython.display import clear_output

# Определение путей
if os.path.exists("/content"):
    BASE_DIR = "/content"
    ROOT_DIR = "/"
else:
    BASE_DIR = os.getcwd()
    ROOT_DIR = "./"

uioawhd = "https://github.com/IAHispano/Applio.git"
new_name = "program_ml"
A = os.path.join(BASE_DIR, "Cache.tar.gz")

# Клонирование репозитория
if not os.path.exists(new_name):
    !git clone --depth 1   --branch exp/vocoders --single-branch

# Переход в папку проекта
if os.path.exists(new_name):
    %cd 
else:
    print(f"ОШИБКА: Папка {new_name} не найдена!")

def vidal_setup():
    if not os.path.exists(A):
        M = os.path.dirname(A)
        if M and not os.path.exists(M):
            os.makedirs(M, exist_ok=True)
        print("Кэшированная установка не найдена..")
        try:
            N = "https://huggingface.co/IAHispano/Applio/resolve/main/Environment/Colab/Cache.tar.gz"
            if os.environ.get("TESTING") == "true":
                print("Тестовый режим: пропускаем загрузку кэша")
                return
            subprocess.run(["wget", "-O", A, N])
            print("Загрузка завершена успешно!")
        except Exception as H:
            print(str(H))
            if os.path.exists(A):
                os.remove(A)
    if Path(A).exists():
        try:
            with tarfile.open(A, "r:gz") as I:
                I.extractall(ROOT_DIR)
                print(f"Распаковка {A} в {ROOT_DIR} завершена.")
            if os.path.exists(A):
                os.remove(A)
        except Exception as e:
            print(f"Ошибка при распаковке: {e}")

vidal_setup()

if os.environ.get("TESTING") != "true":
    !pip uninstall torch torchvision torchaudio -y
    !pip install pydantic==2.8.2 fastapi==0.112.0 starlette==0.37.2
    !pip install -r requirements.txt
    !pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --upgrade --index-url https://download.pytorch.org/whl/cu121
else:
    if os.path.exists("requirements.txt"):
        print("requirements.txt найден.")
    else:
        # If we are in the repo root during test, requirements.txt might be there
        if os.path.exists("../requirements.txt"):
             print("requirements.txt найден в родительской папке.")
        else:
             print(f"ОШИБКА: requirements.txt не найден! Текущая папка: {os.getcwd()}")
             # Note: In reality, git clone would have created it in program_ml/

clear_output()
print("Установка зависимостей завершена!")

In [ ]:
# @title **Запуск Applio**
# @markdown ### Активируйте это только в том случае, если ссылка Gradio не работает
import threading
import urllib.request
import time
import ipywidgets as widgets
from IPython.display import display
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

share_tunnel = False  # @param {type:"boolean"}

def start_applio():
    if share_tunnel:
        !python app.py --listen
    else:
        !python app.py --listen --share

if os.environ.get("TESTING") != "true":
    !npm install -g localtunnel &>/dev/null
    
    %load_ext tensorboard
    %reload_ext tensorboard
    %tensorboard --logdir logs --bind_all

    thread_applio = threading.Thread(target=start_applio)
    thread_applio.start()

    if share_tunnel:
        time.sleep(10)
        get_ipython().system_raw("lt --port 6969 >> url.txt 2>&1 &")
        time.sleep(4)
        try:
            endpoint_ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode("utf8").strip()
            with open("url.txt", "r") as file:
                tunnel_url = file.read().replace("your url is: ", "")
            print(f"Share Link: {tunnel_url}")
            display(widgets.Text(value=endpoint_ip, description="Password IP:", disabled=True))
        except:
            print("Не удалось запустить localtunnel")

    while True:
        time.sleep(5)
else:
    print("Тестовый режим: запуск сервера пропущен")

### **Дополнительно**
Наслаждайтесь дополнительными опциями, которые могут облегчить использование Applio

In [ ]:
# @title Автоматическое резервное копирование
# @markdown При запуске будет активировано или деактивировано для запуска вместе с Applio.
import os
import shutil
import time

LOGS_FOLDER = "logs/"
GOOGLE_DRIVE_PATH = "/content/drive/MyDrive/ApplioBackup"

if "autobackups" not in globals():
    autobackups = False

cooldown = 15  # @param {type:"slider", min:0, max:100, step:0}
def backup_files():
    print("Запуск цикла резервного копирования...")
    while True:
        time.sleep(cooldown)

if not autobackups:
    autobackups = True
    print("Автоматическое резервное копирование включено")
else:
    autobackups = False
    print("Автоматическое резервное копирование отключено")

In [ ]:
# @title Настройка формата папок логов
# @markdown Введите точное имя, которое вы указали как имя модели в Applio.
modelname = "My-Project"  # @param {type:"string"}
logs_folder = f"logs/{modelname}/"
import os
folder_renames = {"0_gt_wavs": "sliced_audios", "1_16k_wavs": "sliced_audios_16k", "2a_f0": "f0", "2b-f0nsf": "f0_voiced", "3_feature768": "v2_extracted"}
if os.path.exists(logs_folder):
    for old_name, new_name in folder_renames.items():
        old_path = os.path.join(logs_folder, old_name)
        new_path = os.path.join(logs_folder, new_name)
        if os.path.exists(old_path):
            os.rename(old_path, new_path)
            print(f"Переименовано: {old_path} -> {new_path}")

In [ ]:
# @title Загрузить резервную копию
# @markdown Введите точное имя, которое вы указали как имя модели в Applio.
import os, shutil
modelname = "My-Project"  # @param {type:"string"}
source_path = "/content/drive/MyDrive/ApplioBackup/" + modelname
destination_path = "logs/" + modelname
if not os.path.exists(source_path):
    print("Папка модели не существует на Google Диске.")
else:
    shutil.copytree(source_path, destination_path, dirs_exist_ok=True)
    print("Резервная копия успешно загружена.")

In [ ]:
# @title Скачать все пользовательские предобученные модели
import os
import urllib.request
out_dir = "rvc/models/pretraineds/pretraineds_custom"
os.makedirs(out_dir, exist_ok=True)
urls = ["https://huggingface.co/ORVC/Ov2Super/resolve/main/f0Ov2Super32kG.pth"]
if os.environ.get("TESTING") != "true":
    for url in urls:
        urllib.request.urlretrieve(url, os.path.join(out_dir, os.path.basename(url)))
    print("Загрузка завершена.")
else:
    print("Тестовый режим: пропуск загрузки моделей")